在 PyCharm 打开本项目后，在窗口右下角添加解释器：添加新的解释器 -> 添加本地解释器 -> 环境：选择现有 -> 类型：Conda -> 环境：lidar（在前期准备中配置好的环境）

In [2]:
# 打开代码后，Tools 文件就在同目录下，所以不用设置 Tools 路径
from Tools.ReadPointCloud import *
import open3d as o3d
from Tools.Treeseg import WatershedSeg
from Tools.Down_Sample import *
from Tools.GroundSegment import *
from Tools.Clean import *
from Tools.Treeseg import *
from Tools.TLS2tree import TLSSeg,MergePoint
from lewos import lewos
import os
import numpy as np

导入数据：

In [4]:
# 点云数据路径
datapth = r"..\LidarData\Faro_DAG_clip.las"
spth = r'.\Output'

# 基于 laspy 读取点云，并转成 open3D 格式
las = ReadPointCloud(datapth)
pcd = laspy2o3d(las)

Read point cloud: ..\LidarData\Faro_DAG_clip.las


点云抽析（下采样）

**注意 `o3d.io.write_point_cloud` 不能覆盖写入，重新运行时需要先将原文件删除。**

In [5]:
# 点云下采样，单位 m
voxel_size = 0.02
pcd_down_sample = voxel_down_sample(pcd, voxel_size=voxel_size)
os.makedirs(spth, exist_ok=True)
o3d.io.write_point_cloud(os.path.join(spth,'Down_sample.pcd'), pcd_down_sample)

Point Cloud Subsample: 0.02


True

点云去噪：
- **方法一** 球形边界去噪 `radius_outlier_removal`
  - `nb_points`：邻居点数阈值
  - `radius`：检测球体半径
- **方法二** 统计学方法去噪 `statistical_outlier_removal`
  - `neighbors`：每个点周围检测的临近点数
  - `ratio`：几倍标准差

In [6]:
# 方法一
pcd_radius_outlier, pcd_radius_inlier = radius_outlier_removal(pcd_down_sample, nb_points=5, radius=0.1)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_radius_outlier.pcd'), pcd_radius_outlier)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_radius_inlier.pcd'), pcd_radius_inlier)

Set outliers (red) and inliers (gray)
----outlier_cloud:  std::vector<Eigen::Vector3d> with 14590 elements.
Use numpy.asarray() to access data.
----inlier_cloud:  std::vector<Eigen::Vector3d> with 610968 elements.
Use numpy.asarray() to access data.


True

In [7]:
# 方法二
pcd_statistical_outlier, pcd_statistical_inlier = statistical_outlier_removal(pcd_down_sample, neighbors=5, ratio=10.0)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_stats_outlier.pcd'), pcd_statistical_outlier)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_stats_inlier.pcd'), pcd_statistical_inlier)

Set outliers (red) and inliers (gray)
----outlier_cloud:  std::vector<Eigen::Vector3d> with 719 elements.
Use numpy.asarray() to access data.
----inlier_cloud:  std::vector<Eigen::Vector3d> with 624839 elements.
Use numpy.asarray() to access data.


True

地面点分离，先提取 z 值最低点，再对这些点采用泊松表面重建算法拟合地面。

In [8]:
# 如果之前的结果在重启 kernel 之后消失了，就需要重新读入
# 选择用 pcd_statistical_inlier 或者 pcd_radius_inlier 都可以
# pcd_radius_inlier = o3d.io.read_point_cloud(os.path.join(spth,'pcd_radius_inlier.pcd'))
# pcd_statistical_inlier = o3d.io.read_point_cloud(os.path.join(spth,'pcd_stats_inlier.pcd'))

# ground, aboveground = PlaneSegmentation(pcd)  # 直接拟合点云中的平面并分割，我们一般不使用这种方法寻找地面点
ground, aboveground = PoissonSurfaceFilter(pcd_radius_inlier, pillar_size=1, threshold=0.3)  # 寻找最低点 + 泊松表面重建算法
o3d.io.write_point_cloud(os.path.join(spth,'ground.pcd'), ground)
o3d.io.write_point_cloud(os.path.join(spth,'aboveground.pcd'), aboveground)

Find the lowest point in pillars
run Poisson surface reconstruction
[Open3D DEBUG] Input Points / Samples: 63 / 63
[Open3D DEBUG] #   Got kernel density: 0.0019998550415039062 (s), 1127.8828125 (MB) / 1127.8828125 (MB) / 2055 (MB)
[Open3D DEBUG] #     Got normal field: 0.004000186920166016 (s), 1129.80859375 (MB) / 1129.80859375 (MB) / 2055 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 3.110358e-02 / 1.959526e+00
[Open3D DEBUG] #       Finalized tree: 0.006999969482421875 (s), 1135.48828125 (MB) / 1135.48828125 (MB) / 2055 (MB)
[Open3D DEBUG] #  Set FEM constraints: 0.03900003433227539 (s), 1135.74609375 (MB) / 1135.74609375 (MB) / 2055 (MB)
[Open3D DEBUG] #Set point constraints: 0.0009999275207519531 (s), 1135.75390625 (MB) / 1135.75390625 (MB) / 2055 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 76644 / 66888 / 20705
[Open3D DEBUG] Memory Usage: 1135.754 MB
[Open3D DEBUG] # Linear system solved: 0.09299993515014648 (s), 1137.95703125 (MB) / 1137.95703125 (MB) / 20

True

利用 LeWoS 算法进行枝叶分离。

In [9]:
# 基于 LeWoS 进行枝叶分离
import os
import numpy as np
import open3d as o3d
from lewos import lewos

points = np.asarray(aboveground.points)
labels = lewos(points, ft_threshold=0.125, paral=8)

# 保存坐标和标签，供后续 MergePoint 使用
# 四列：X Y Z label；1 = 枝干，0 = 叶片
os.makedirs(spth, exist_ok=True)
np.savetxt(
    os.path.join(spth, "leafwood.txt"),
    np.column_stack((points, labels)),
    fmt=["%.8f", "%.8f", "%.8f", "%d"],
)

# 提取枝干和叶片，保留 Open3D 点云格式
wood = aboveground.select_by_index(np.flatnonzero(labels == 1).tolist())
leaf = aboveground.select_by_index(np.flatnonzero(labels == 0).tolist())

# 着色
wood.paint_uniform_color([0.55, 0.30, 0.10])
leaf.paint_uniform_color([0.20, 0.75, 0.20])

# 保存分类后的点云
o3d.io.write_point_cloud(os.path.join(spth, "wood.pcd"), wood)
o3d.io.write_point_cloud(os.path.join(spth, "leaf.pcd"), leaf)

# 可视化
o3d.visualization.draw_geometries(
    [wood, leaf],
    window_name="LeWoS 枝叶分离",
    width=1200,
    height=800,
)

单木分割算法原理简述：
1. 去除地形。
2. 按照离地面高度切割树干层（默认 0.2 m 一层，可调），每层内部用 DBSCAN 算法寻找连通点簇，DBSCAN 参数暂时不可调。
3. 寻找主干候选对象并进行圆柱拟合，默认选取离地约 1.5～2.0 m 的木质点，高度可调。然后再次 DBSCAN 将同一树干不同切片的点合并为候选对象。
4. 对每个主干候选对象做 RANSAC 圆柱拟合
5. 用最短路径算法为符合要求的圆柱主干分配树枝：图结点为每个联通点簇，边权为足够近的两个点簇之间的最短距离，在图上进行 Dijkstra 算法寻找每个点簇的最近主干，即为该点簇的归属。
6. 为叶片和未分配的枝干点分配树木，这一步以体素为单位，每个体素连接至最近的树枝，并归属到该树枝所在的树木中。

下面代码实现单木分割。

In [10]:
import sys
sys.argv = ['']

#############################################
# MKL loads additional DLLs at runtime using PATH. Set it before NumPy/SciPy imports.
# 修复可能出现的动态链接库路径丢失问题
# 这段代码可以先注释掉，如果运行出现问题再加上
import faulthandler
_conda_bin = os.path.join(sys.prefix, 'Library', 'bin')
if os.name == 'nt' and os.path.isdir(_conda_bin):
    os.environ['PATH'] = _conda_bin + os.pathsep + os.environ.get('PATH', '')
faulthandler.enable()
#############################################

pcd = MergePoint(groundpth=os.path.join(spth,'ground.pcd'),leafwoodpth=os.path.join(spth,'leafwood.txt'))
TLSSeg(pcd,spth, args=[
    '--slice-thickness', '0.2',  # 枝干点分层厚度，单位 m
    '--find-stems-boundary', '1.5', '2.0',  # 寻找候选树干的离地高度范围，单位 m
    '--find-stems-min-radius', '0.05',  # 接受主干的最小半径，单位 m
    '--find-stems-min-points', '50',  # 主干点数必须大于此值
    '--graph-edge-length', '0.5',  # 枝干点簇之间允许的最大连接距离，单位 m；过大容易连接到邻树。
    '--graph-maximum-cumulative-gap', 'inf',  # 枝干点簇到树干的累计图距离上限，单位 m；inf 表示不限制
    '--min-points-per-tree', '0',  # 已分配的枝干点数必须大于此值，才保留这棵树
    '--add-leaves-voxel-length', '0.5',  # 补叶阶段的体素边长，单位 m
    # '--add-leaves',  # 执行补叶；当前默认已开启，目前无法关闭。
    # '--verbose',  # 打印运行信息和进度；当前默认已开启，目前无法关闭。
])

Merge Point Cloud ....
---- parameters ----
odir                               .\Output
tindex                             None
n_tiles                            3
n_zeros                            3
overlap                            False
slice_thickness                    0.2
find_stems_boundary                [1.5, 2.0]
find_stems_min_radius              0.05
find_stems_min_points              50
graph_edge_length                  0.5
graph_maximum_cumulative_gap       inf
min_points_per_tree                0
add_leaves                         1
add_leaves_voxel_length            0.5
add_leaves_edge_length             1
save_diameter_class                False
ignore_missing_tiles               False
pandarallel                        False
verbose                            1
Making dtm...

----- skeletonisation started -----


slice data vertically and clustering: 100%|██████████| 73/73 [00:00<00:00, 164.41it/s]


fitting convex hulls to clusters
identifying stems...
fitting cylinders to possible stems...
generating graph, this may take a while...
merging skeleton points with graph


writing stems to file: 100%|██████████| 6/6 [00:00<00:00, 3166.71it/s]

adding leaves to stems, this may take a while...



100%|██████████| 6/6 [00:00<00:00, 103.88it/s]


runtime: 132
